# Test Kaggle Models API

Verify the CLI commands for uploading/downloading model checkpoints.

In [ ]:
# Cell 1 -- Config and dummy checkpoint

import os, json, subprocess, tarfile

MODEL_OWNER = "mattfletcherbund"
MODEL_SLUG = "test-checkpoint-api"
MODEL_HANDLE = f"{MODEL_OWNER}/{MODEL_SLUG}/pytorch/default"
UPLOAD_DIR = "/kaggle/working/model-upload"
DOWNLOAD_DIR = "/kaggle/working/model-download"

os.makedirs(UPLOAD_DIR, exist_ok=True)

# Write a small dummy file as our "resume.pt"
with open(os.path.join(UPLOAD_DIR, "resume.pt"), "wb") as f:
    f.write(b"dummy checkpoint data")

print(f"Created dummy resume.pt ({os.path.getsize(os.path.join(UPLOAD_DIR, 'resume.pt'))} bytes)")

In [ ]:
# Cell 2 -- Create the model (if it doesn't exist)

import json, os, subprocess

result = subprocess.run(
    ["kaggle", "models", "list", "-m", "-s", MODEL_SLUG],
    capture_output=True, text=True,
)

if MODEL_SLUG in result.stdout:
    print(f"Model '{MODEL_SLUG}' already exists, skipping creation.")
else:
    model_meta = {
        "ownerSlug": MODEL_OWNER,
        "title": "Test Checkpoint API",
        "slug": MODEL_SLUG,
        "subtitle": "Testing model upload/download",
        "isPrivate": True,
        "description": "Temporary model for testing Kaggle Models API",
        "publishTime": "",
        "provenanceSources": "",
    }

    with open(os.path.join(UPLOAD_DIR, "model-metadata.json"), "w") as f:
        json.dump(model_meta, f, indent=2)

    !kaggle models create -p {UPLOAD_DIR}

In [ ]:
# Cell 3 -- Create the model instance (if it doesn't exist)

import json, os, subprocess

result = subprocess.run(
    ["kaggle", "models", "instances", "list", f"{MODEL_OWNER}/{MODEL_SLUG}"],
    capture_output=True, text=True,
)

if "default" in result.stdout:
    print("Instance 'default' already exists, skipping creation.")
else:
    instance_meta = {
        "ownerSlug": MODEL_OWNER,
        "modelSlug": MODEL_SLUG,
        "instanceSlug": "default",
        "framework": "pytorch",
        "overview": "Training checkpoint",
        "licenseName": "Apache 2.0",
        "fineTunable": False,
    }

    with open(os.path.join(UPLOAD_DIR, "model-instance-metadata.json"), "w") as f:
        json.dump(instance_meta, f, indent=2)

    !kaggle models instances create -p {UPLOAD_DIR}

In [ ]:
# Cell 4 -- Upload a version (if none exists yet)

import os, subprocess

# Check if any versions exist already
result = subprocess.run(
    ["kaggle", "models", "instances", "versions", "list", MODEL_HANDLE],
    capture_output=True, text=True,
)

version_lines = [l for l in result.stdout.strip().split("\n")[1:] if l.strip()]
if version_lines:
    print(f"Version {version_lines[0].split()[0]} already exists, skipping upload.")
else:
    # Write a dummy file to upload
    with open(os.path.join(UPLOAD_DIR, "resume.pt"), "wb") as f:
        f.write(b"dummy checkpoint data v2")

    !kaggle models instances versions create {MODEL_HANDLE} -p {UPLOAD_DIR} -n "Test version"

In [ ]:
# Cell 5 -- Download the latest version

import os, subprocess, tarfile

DOWNLOAD_DIR = "/kaggle/working/model-download"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Get latest version number
result = subprocess.run(
    ["kaggle", "models", "instances", "versions", "list", MODEL_HANDLE],
    capture_output=True, text=True,
)
print(result.stdout)
latest_version = result.stdout.strip().split("\n")[1].split()[0]
print(f"Latest version: {latest_version}")

# Download with explicit version
VERSIONED_HANDLE = f"{MODEL_HANDLE}/{latest_version}"
!kaggle models instances versions download {VERSIONED_HANDLE} -p {DOWNLOAD_DIR}

# Check what we got
!ls -la {DOWNLOAD_DIR}/
for f in os.listdir(DOWNLOAD_DIR):
    if f.endswith(".tar.gz"):
        with tarfile.open(os.path.join(DOWNLOAD_DIR, f)) as tar:
            tar.extractall(DOWNLOAD_DIR)
        os.remove(os.path.join(DOWNLOAD_DIR, f))
        print(f"Extracted {f}")

# Verify
!ls -la {DOWNLOAD_DIR}/
if os.path.isfile(os.path.join(DOWNLOAD_DIR, "resume.pt")):
    with open(os.path.join(DOWNLOAD_DIR, "resume.pt"), "rb") as f:
        print(f"Downloaded content: {f.read()}")
else:
    print("resume.pt not found -- check directory listing above")

In [ ]:
# Cell 6 -- Clean up: delete the test model

# Uncomment to delete:
# !kaggle models delete {MODEL_OWNER}/{MODEL_SLUG} -y